# Fashion MNIST Classification Project

## Project Overview
The objective of this project is to build a convolutional neural network (CNN) using PyTorch to classify grayscale clothing images from the Fashion-MNIST dataset. The workflow covers data loading, preprocessing, visualization, CNN construction, training, validation, testing, and performance evaluation.

**Classes:** T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

## 1. Import Libraries and Set the Device

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)

## 2. Load and Preprocess the Fashion-MNIST Dataset

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print('Training samples:', len(train_dataset))
print('Testing samples:', len(test_dataset))
print('Image shape:', train_dataset[0][0].shape)

## 3. Define the Clothing Classes

In [3]:
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')
print('Number of classes:', len(classes))
print(classes)

## 4. Visualize Sample Images

In [4]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image = images[i].squeeze().numpy()
    image = image * 0.5 + 0.5
    ax.imshow(image, cmap='gray')
    ax.set_title(classes[labels[i].item()])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Build the Convolutional Neural Network

In [5]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = FashionCNN().to(device)
print(model)

## 6. Define Loss Function and Optimizer

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print('Loss function:', criterion)
print('Optimizer:', optimizer.__class__.__name__)


## 7. Train the CNN

In [7]:
num_epochs = 5
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    print(f'Epoch {epoch + 1}/{num_epochs} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_accuracy:.2f}%')

## 8. Plot Training Performance

In [8]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_accuracies, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy')
plt.tight_layout()
plt.show()

## 9. Evaluate the Model on the Test Set

In [9]:
model.eval()
correct = 0
total = 0
all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100.0 * correct / total
print(f'Test Accuracy: {test_accuracy:.2f}%')

## 10. Confusion Matrix

In [10]:
confusion_matrix = torch.zeros(10, 10, dtype=torch.int64)
for true_label, predicted_label in zip(all_labels, all_predictions):
    confusion_matrix[true_label, predicted_label] += 1

plt.figure(figsize=(8, 7))
plt.imshow(confusion_matrix.numpy(), cmap='Blues')
plt.colorbar()
plt.xticks(range(10), range(10))
plt.yticks(range(10), range(10))
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Fashion-MNIST Confusion Matrix')
plt.show()

## 11. Display Example Predictions

In [11]:
model.eval()
sample_images, sample_labels = next(iter(test_loader))
with torch.no_grad():
    sample_outputs = model(sample_images.to(device))
    sample_predictions = sample_outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image = sample_images[i].squeeze().numpy() * 0.5 + 0.5
    ax.imshow(image, cmap='gray')
    ax.set_title(f'True: {classes[sample_labels[i]]}\nPred: {classes[sample_predictions[i]]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 12. Conclusion

A convolutional neural network was developed in PyTorch for Fashion-MNIST image classification. The model uses convolutional layers to learn local visual patterns, max-pooling to reduce spatial dimensions, dropout to reduce overfitting, and fully connected layers for final classification. The final test accuracy is calculated above from the held-out Fashion-MNIST test set.

This project demonstrates a complete deep-learning workflow: preprocessing, CNN architecture design, training, visualization, testing, and error analysis.